# 01 · 데이터 전처리 (Preprocessing)

원본 CSV → **상수 컬럼 제거 → 결측치 처리 → 전압/온도 208개 추출** → `./data/preprocessed/...` 저장.
학습·테스트 노트북은 여기서 저장한 CSV만 사용한다.

**사용법**: 아래 `설정` 셀의 경로/컬럼 범위만 바꿔 원본 파일마다 한 번씩 실행.
학습(1000_chg)과 테스트는 상수 제거 결과가 달라 전압/온도 컬럼 인덱스 범위가 다르다(원본 반영).

In [35]:
from function_def import *
import os, pandas as pd

## 설정 (파일마다 바꿔 재실행)

In [ ]:
# --- 학습 데이터 예시 ---
RAW_PATH = './data/raw_data/test/Test03_OK_chg .csv'
OUT_PATH = './data/preprocessed/test/Test03_OK_chg .csv'
VT_START, VT_END = 4, 226

# --- 테스트로 돌릴 때 (예: Test07) ---
# RAW_PATH = './data/raw_data/test/Test07_NG_dchg.csv'
# OUT_PATH = './data/preprocessed/test/Test07_NG_dchg.csv'
# VT_START, VT_END = 1, 209

print("RAW:", RAW_PATH); print("OUT:", OUT_PATH)

RAW: ./data/raw_data/test/Test03_OK_chg.csv
OUT: ./data/preprocessed/test/Test03_OK_chg.csv


## 1) 로드 & 요약 확인

In [39]:
data = pd.read_csv(RAW_PATH)
print("shape(행,열):", data.shape, "| 결측치:", int(data.isna().sum().sum()))
data.head(5)

for i in range(len(data.columns)):
    if i < VT_START or i > VT_END:
       print(i,data.columns[i])

FileNotFoundError: [Errno 2] No such file or directory: './data/raw_data/test/Test03_OK_chg.csv'

In [15]:
data.columns

Index(['Date', 'Time', 'SerialNumber', 'Voltage', 'Current', 'RSOCmin',
       'RSOCmax', 'RSOCavg', 'USOCmin', 'USOCmax',
       ...
       'M12T01', 'M12T02', 'M13T01', 'M13T02', 'M14T01', 'M14T02', 'M15T01',
       'M15T02', 'M16T01', 'M16T02'],
      dtype='object', length=231)

## 2) 상수 컬럼 제거

In [16]:

data1 = removeConstant(data, 1)
print("제거된 컬럼:", len(set(data.columns)-set(data1.columns)), "| 후 shape:", data1.shape)

제거된 컬럼: 22 | 후 shape: (1723, 209)


In [17]:
print("\n--- 제거된 컬럼별 상세 정보 ---") #제거된 칼럼의 정보 출력하는 블록
for col in sorted(set(data.columns)-set(data1.columns)):
    # nunique(): 해당 컬럼의 고유값 개수 (제거 대상이므로 보통 1이 나와야 정상)
    n_unique = data[col].nunique()

    # unique(): 실제로 어떤 값(들)으로 채워져 있었는지 배열로 반환.
    # 상수 컬럼이면 원소가 1개짜리 배열이 나온다. 예: array([0.])
    unique_val = data[col].unique()

    # isnull().mean(): 결측 비율. 혹시 "값이 거의 다 NaN이고 딱 1개 값만 있는" 경우인지
    # 구분해서 보기 위함 -> 센서 고장(항상 같은 값) vs 데이터 누락(대부분 NaN)을 구별하는 단서
    missing_ratio = data[col].isnull().mean()

    print(f"{col:15s} | 고유값 개수: {n_unique} | 값: {unique_val} | 결측비율: {missing_ratio:.1%}")


--- 제거된 컬럼별 상세 정보 ---
ChgImax         | 고유값 개수: 1 | 값: [0] | 결측비율: 0.0%
ChgPmax         | 고유값 개수: 1 | 값: [0] | 결측비율: 0.0%
Current         | 고유값 개수: 1 | 값: [0] | 결측비율: 0.0%
DV              | 고유값 개수: 1 | 값: [0] | 결측비율: 0.0%
Date            | 고유값 개수: 1 | 값: ['2021-05-27'] | 결측비율: 0.0%
DchgImax        | 고유값 개수: 1 | 값: [0] | 결측비율: 0.0%
DchgPmax        | 고유값 개수: 1 | 값: [0] | 결측비율: 0.0%
Power           | 고유값 개수: 1 | 값: [0] | 결측비율: 0.0%
RSOCavg         | 고유값 개수: 1 | 값: [0] | 결측비율: 0.0%
RSOCmax         | 고유값 개수: 1 | 값: [0] | 결측비율: 0.0%
RSOCmin         | 고유값 개수: 1 | 값: [0] | 결측비율: 0.0%
SOH             | 고유값 개수: 1 | 값: [0] | 결측비율: 0.0%
SerialNumber    | 고유값 개수: 1 | 값: [166] | 결측비율: 0.0%
Tavg            | 고유값 개수: 1 | 값: [0] | 결측비율: 0.0%
Tmax            | 고유값 개수: 1 | 값: [0] | 결측비율: 0.0%
Tmin            | 고유값 개수: 1 | 값: [0] | 결측비율: 0.0%
USOCavg         | 고유값 개수: 1 | 값: [0] | 결측비율: 0.0%
USOCmax         | 고유값 개수: 1 | 값: [0] | 결측비율: 0.0%
USOCmin         | 고유값 개수: 1 | 값: [0] | 결측비율: 0.0%
Vmax          

## 3) 결측치 처리 (시계열: 버리지 않고 채움)

In [18]:
data2 = handleMissingValue(data1)
print("결측치 처리 후 shape:", data2.shape)

결측치 처리 후 shape: (1723, 209)


In [19]:
data2.columns

Index(['Time', 'M01CV01', 'M01CV02', 'M01CV03', 'M01CV04', 'M01CV05',
       'M01CV06', 'M01CV07', 'M01CV08', 'M01CV09',
       ...
       'M12T01', 'M12T02', 'M13T01', 'M13T02', 'M14T01', 'M14T02', 'M15T01',
       'M15T02', 'M16T01', 'M16T02'],
      dtype='object', length=209)

## 4) 전압/온도 컬럼 추출 (인덱스 범위 점검)

In [20]:
VT_START, VT_END = 1, 226
final_data_0 = data2
columns_name = data2.columns[VT_START:VT_END]
final_data = final_data_0[columns_name]
for i, c in enumerate(final_data):
    print(i, c)

0 M01CV01
1 M01CV02
2 M01CV03
3 M01CV04
4 M01CV05
5 M01CV06
6 M01CV07
7 M01CV08
8 M01CV09
9 M01CV10
10 M01CV11
11 M02CV01
12 M02CV02
13 M02CV03
14 M02CV04
15 M02CV05
16 M02CV06
17 M02CV07
18 M02CV08
19 M02CV09
20 M02CV10
21 M02CV11
22 M03CV01
23 M03CV02
24 M03CV03
25 M03CV04
26 M03CV05
27 M03CV06
28 M03CV07
29 M03CV08
30 M03CV09
31 M03CV10
32 M03CV11
33 M04CV01
34 M04CV02
35 M04CV03
36 M04CV04
37 M04CV05
38 M04CV06
39 M04CV07
40 M04CV08
41 M04CV09
42 M04CV10
43 M04CV11
44 M05CV01
45 M05CV02
46 M05CV03
47 M05CV04
48 M05CV05
49 M05CV06
50 M05CV07
51 M05CV08
52 M05CV09
53 M05CV10
54 M05CV11
55 M06CV01
56 M06CV02
57 M06CV03
58 M06CV04
59 M06CV05
60 M06CV06
61 M06CV07
62 M06CV08
63 M06CV09
64 M06CV10
65 M06CV11
66 M07CV01
67 M07CV02
68 M07CV03
69 M07CV04
70 M07CV05
71 M07CV06
72 M07CV07
73 M07CV08
74 M07CV09
75 M07CV10
76 M07CV11
77 M08CV01
78 M08CV02
79 M08CV03
80 M08CV04
81 M08CV05
82 M08CV06
83 M08CV07
84 M08CV08
85 M08CV09
86 M08CV10
87 M08CV11
88 M09CV01
89 M09CV02
90 M09CV03
91 M09CV0

In [9]:
#columns_vt = data2.columns[VT_START:VT_END]
#print("추출 컬럼 수:", len(columns_vt))
#df_out = data2[columns_vt]
#df_out.head(5)

## 5) 저장

In [21]:
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
final_data.to_csv(OUT_PATH, index=False)
print("저장 완료:", OUT_PATH, "| shape:", final_data.shape)

저장 완료: ./data/preprocessed/test/test01_OK_chg.csv | shape: (1723, 208)


## +@다중 파일 전처리 및 PCA

In [24]:
import os
import numpy as np
import pandas as pd
import joblib
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, MinMaxScaler   # 스케일러 2종
from sklearn.impute import SimpleImputer                          # 결측 대치 (모듈이 preprocessing이 아니라 impute)
from function_def import *
from parameter import CFG

diffs_n, lags_n, smooth_n = CFG['diffs_n'], CFG['lags_n'], CFG['smooth_n']
# ---------------------------------------------------------------
# 0. 파일 목록 생성: 1000 ~ 1050 (총 51개)
#    - 손으로 51줄 적으면 오타 위험 → range로 규칙 생성
#    - range(1000, 1051): 끝값 1051은 미포함 → 실제로 1000..1050 = 51개
# ---------------------------------------------------------------
DATA_DIR = './data/raw_data/train'
file_ids = range(1000, 1051)
files = [os.path.join(DATA_DIR, f'{fid}_chg.csv') for fid in file_ids]

# 0-1. 존재 확인: 없는 파일은 걸러내고 경고 (파일 하나가 조용히 빠지는 것 방지)
missing = [f for f in files if not os.path.exists(f)]
if missing:
    print(f"[경고] 없는 파일 {len(missing)}개:", [os.path.basename(m) for m in missing])
files = [f for f in files if os.path.exists(f)]
print(f"불러올 파일 수: {len(files)}개")
assert len(files) > 0, "불러올 파일이 없습니다. 경로를 확인하세요."

# ---------------------------------------------------------------
# 1. 각 파일 읽기 + 컬럼 정합성 검사 (concat 전에 반드시)
#    - 51개 중 하나라도 컬럼 구성/순서가 다르면 pd.concat이 에러 대신
#      NaN을 채우거나 열을 어긋나게 정렬 → StandardScaler/PCA fit이 통째로 오염됨
#    - 에러가 안 나서 학습이 다 끝난 뒤에야 이상 증상으로 드러나는 최악의 버그
# ---------------------------------------------------------------
raw_dfs = [pd.read_csv(f) for f in files]

ref_cols = list(raw_dfs[0].columns)          # 첫 파일을 기준 컬럼으로
for f, df in zip(files, raw_dfs):
    if list(df.columns) != ref_cols:         # 이름뿐 아니라 순서까지 동일해야 PCA 축이 안 섞임
        raise ValueError(
            f"컬럼 불일치: {os.path.basename(f)}\n"
            f"  기준: {ref_cols}\n  실제: {list(df.columns)}"
        )

# 검사 통과분에만 diff_smooth 적용 (파일별로 따로)
dfs = [diff_smooth_df(df, lags_n, diffs_n, smooth_n) for df in raw_dfs]
print("각 파일 shape:", [d.shape for d in dfs])



불러올 파일 수: 51개
각 파일 shape: [(6009, 231), (6009, 231), (6008, 231), (6008, 231), (8515, 231), (8514, 231), (4796, 231), (4795, 231), (6055, 231), (6049, 231), (6092, 231), (6093, 231), (4982, 231), (8240, 231), (8408, 231), (8334, 231), (5022, 231), (7246, 231), (10454, 231), (10914, 231), (11008, 231), (10998, 231), (2370, 231), (1069, 231), (1685, 231), (1805, 231), (1285, 231), (1044, 231), (1056, 231), (1752, 231), (1240, 231), (9609, 231), (6620, 231), (5061, 231), (8618, 231), (7271, 231), (11316, 231), (11024, 231), (11270, 231), (11008, 231), (10998, 231), (1711, 231), (2369, 231), (12582, 231), (12110, 231), (12005, 231), (11955, 231), (1069, 231), (1685, 231), (1044, 231), (2654, 231)]


In [34]:
import os
import numpy as np
import pandas as pd

# ===============================================================
# 설정값 (여기만 바꾸면 됨)
# ===============================================================
DATA_DIR = './data/raw_data/train'          # 원본 폴더
OUT_DIR = './data/preprocessed/train'      # 정리본 저장 폴더
FILE_IDS = range(1000, 1051)                     # 1000~1050 = 51개 (끝값 1051은 미포함)
SLICE_START_COL = 'M01CV01'                      # 이 컬럼부터 마지막까지만 남김


# ===============================================================
# 1) 원본 파일 일괄 로드
# ===============================================================
def load_raw_files(data_dir, file_ids):
    """{파일명: DataFrame} 딕셔너리로 모든 원본을 읽어온다."""
    files = [os.path.join(data_dir, f'{fid}_chg.csv') for fid in file_ids]

    # 존재 확인: 없는 파일은 걸러내고 경고 (파일 하나가 조용히 빠지는 것 방지)
    missing = [f for f in files if not os.path.exists(f)]
    if missing:
        print(f'[경고] 없는 파일 {len(missing)}개:', [os.path.basename(m) for m in missing])
    files = [f for f in files if os.path.exists(f)]
    assert files, '불러올 파일이 하나도 없습니다. DATA_DIR/FILE_IDS를 확인하세요.'

    raw = {os.path.basename(f): pd.read_csv(f) for f in files}
    print(f'불러온 파일 수: {len(raw)}개')
    return raw


# ===============================================================
# 2) 원본 단계 컬럼 정합성 사전검사
#    - 전역 상수판정을 하려면 concat이 필요한데, 원본 컬럼이 이미 다르면
#      concat이 NaN을 채워 판정을 오염시킴 → 시작부터 막는다.
# ===============================================================
def assert_raw_columns_match(raw):
    ref_name = next(iter(raw))                   # 첫 파일을 기준으로
    ref_cols = list(raw[ref_name].columns)
    for name, df in raw.items():
        if list(df.columns) != ref_cols:         # 이름 + 순서까지 동일해야 함
            raise ValueError(
                f'원본 컬럼 불일치: {name}\n  기준({ref_name}): {ref_cols}\n  실제: {list(df.columns)}'
            )
    print(f'원본 컬럼 정합성 OK (컬럼 {len(ref_cols)}개)')
    return ref_cols


# ===============================================================
# 3) 전역 상수 컬럼 탐지
#    - 51개를 세로로 합친 전체 기준으로 고유값이 1개 이하인 컬럼을 찾는다.
#    - nunique(dropna=True): NaN은 세지 않음 → '전부 NaN'인 컬럼도 상수로 간주해 제거.
#    - 이 '동일한 목록'을 모든 파일에 똑같이 적용해야 컬럼 일치가 유지됨.
# ===============================================================
def find_constant_columns(raw):
    concat = pd.concat(raw.values(), axis=0, ignore_index=True)
    nun = concat.nunique(dropna=True)            # 컬럼별 고유값 개수
    const_cols = nun[nun <= 1].index.tolist()    # 1개 이하 = 정보 없음
    print(f'전역 상수 컬럼 {len(const_cols)}개 제거 예정:', const_cols)
    return const_cols


# ===============================================================
# 4) 파일 1개 전처리 — 요청하신 순서 그대로
#    (1) 상수 컬럼 제거  (2) 결측치 제거  (3) M01CV01부터 슬라이싱
# ===============================================================
def preprocess_one(df, const_cols, slice_start_col):
    # (1) 상수 컬럼 제거: 전역으로 정한 동일 목록을 적용 (컬럼 일치 보장의 핵심)
    df = df.drop(columns=[c for c in const_cols if c in df.columns])

    # (2) 결측치 제거: NaN이 든 '행'을 삭제 (행 수는 파일마다 달라도 무방 → 컬럼엔 영향 없음)
    df = df.dropna(axis=0).reset_index(drop=True)

    # (3) 인덱스 슬라이싱: slice_start_col의 위치부터 마지막 컬럼까지만 남김
    if slice_start_col not in df.columns:
        raise KeyError(
            f"'{slice_start_col}' 컬럼이 없습니다. 상수제거로 사라졌거나 원본에 없음.\n"
            f'  현재 컬럼: {list(df.columns)}'
        )
    start = df.columns.get_loc(slice_start_col)  # 컬럼의 정수 위치
    df = df.iloc[:, start:]                       # 그 위치부터 끝까지 (열 슬라이싱)
    return df


# ===============================================================
# 5) 전처리 후 컬럼 일치 검사 (요구사항 2)
# ===============================================================
def check_column_consistency(clean):
    ref_name = next(iter(clean))
    ref_cols = list(clean[ref_name].columns)
    mismatches = {n: list(d.columns) for n, d in clean.items() if list(d.columns) != ref_cols}

    if mismatches:
        print('\n[❌ 컬럼 불일치 발견]')
        print(f'  기준({ref_name}): {ref_cols}')
        for n, cols in mismatches.items():
            print(f'  - {n}: {cols}')
        return False

    print(f'\n[✅ 컬럼 일치] {len(clean)}개 파일 모두 동일 (컬럼 {len(ref_cols)}개)')
    print('  최종 컬럼:', ref_cols)
    return True


# ===============================================================
# 6) 오케스트레이션 (전체 실행)
# ===============================================================
def run(save=True):
    raw = load_raw_files(DATA_DIR, FILE_IDS)
    assert_raw_columns_match(raw)                # 원본 사전검사
    const_cols = find_constant_columns(raw)      # 전역 상수 목록 (한 번만)

    clean = {}
    for name, df in raw.items():
        before = df.shape
        cdf = preprocess_one(df, const_cols, SLICE_START_COL)
        clean[name] = cdf
        print(f'  {name}: {before} → {cdf.shape} (행 {before[0]-cdf.shape[0]}개 결측제거)')

    ok = check_column_consistency(clean)         # 전처리 후 일치 검사

    if save and ok:
        os.makedirs(OUT_DIR, exist_ok=True)
        for name, df in clean.items():
            df.to_csv(os.path.join(OUT_DIR, name), index=False)
        print(f'\n정리본 저장 완료 → {OUT_DIR}')

    return clean, ok


if __name__ == '__main__':
    run()

불러온 파일 수: 51개
원본 컬럼 정합성 OK (컬럼 231개)
전역 상수 컬럼 1개 제거 예정: ['SOH']
  1000_chg.csv: (6009, 231) → (6009, 208) (행 0개 결측제거)
  1001_chg.csv: (6009, 231) → (6009, 208) (행 0개 결측제거)
  1002_chg.csv: (6008, 231) → (6008, 208) (행 0개 결측제거)
  1003_chg.csv: (6008, 231) → (6008, 208) (행 0개 결측제거)
  1004_chg.csv: (8515, 231) → (8515, 208) (행 0개 결측제거)
  1005_chg.csv: (8514, 231) → (8514, 208) (행 0개 결측제거)
  1006_chg.csv: (4796, 231) → (4796, 208) (행 0개 결측제거)
  1007_chg.csv: (4795, 231) → (4795, 208) (행 0개 결측제거)
  1008_chg.csv: (6055, 231) → (6055, 208) (행 0개 결측제거)
  1009_chg.csv: (6049, 231) → (6049, 208) (행 0개 결측제거)
  1010_chg.csv: (6092, 231) → (6092, 208) (행 0개 결측제거)
  1011_chg.csv: (6093, 231) → (6093, 208) (행 0개 결측제거)
  1012_chg.csv: (4982, 231) → (4982, 208) (행 0개 결측제거)
  1013_chg.csv: (8240, 231) → (8239, 208) (행 1개 결측제거)
  1014_chg.csv: (8408, 231) → (8408, 208) (행 0개 결측제거)
  1015_chg.csv: (8334, 231) → (8334, 208) (행 0개 결측제거)
  1016_chg.csv: (5022, 231) → (5022, 208) (행 0개 결측제거)
  1017_chg.csv: (7

In [ ]:
# ---------------------------------------------------------------
# 2. 표준화 + PCA는 "합친 데이터"로 fit (공통 좌표계)
# ---------------------------------------------------------------
df_concat = pd.concat(dfs, axis=0, ignore_index=True)   # 세로로 합치기

scaler_std = StandardScaler()
scaled_concat = scaler_std.fit_transform(df_concat)     # 표준화 (스케일 차이 제거)

features_dim = 3                                         # PCA 축 개수
pca = PCA(n_components=features_dim)
pca.fit(scaled_concat)                                   # 공통 축 학습 (fit만)

# 설명력 확인 (며칠 전 강조한 부분)
print("explained_variance_ratio:", pca.explained_variance_ratio_)
print("누적 설명력:", np.cumsum(pca.explained_variance_ratio_))
# 3개 축이 전체 분산의 몇 %를 설명하는지 반드시 확인

# ---------------------------------------------------------------
# 3. 각 파일을 "따로" 변환 + 윈도우 (파일 경계 오염 방지)
# ---------------------------------------------------------------
X_list = []
for df in dfs:
    # 3-1. 같은 표준화 + 같은 PCA로 변환 (transform만, fit 안 함)
    scaled = scaler_std.transform(df)
    data = pca.transform(scaled)

    # 3-2. date 컬럼 붙이기 (time_segments_aggregate가 요구)
    tmp = pd.DataFrame(data, columns=[f'pca_{i}' for i in range(1, features_dim+1)])
    tmp.insert(0, 'date', range(1, len(tmp)+1))

    # 3-3. aggregate → 결측 처리
    Xf, idxf = time_segments_aggregate(tmp, interval=1, time_column='date')
    Xf = SimpleImputer().fit_transform(Xf)

    # 3-4. MinMax 스케일 (이건 나중에 합쳐서 fit하는 게 이상적 — 아래 주석 참고)
    #      여기선 일단 각자 두고, 합친 뒤 다시 스케일하는 방식으로 감
    X_list.append((Xf, idxf))

# ---------------------------------------------------------------
# 4. MinMax 스케일은 "합친 데이터" 기준으로 fit
#    (모델 입력 범위 -1~1을 51개 파일 전체 기준으로 통일)
# ---------------------------------------------------------------
X_all_raw = np.concatenate([x for x, _ in X_list], axis=0)
scaler_mm = MinMaxScaler(feature_range=(-1, 1))
scaler_mm.fit(X_all_raw)                                 # 합친 기준으로 fit

# ---------------------------------------------------------------
# 5. 각 파일을 스케일 + 윈도우 → 윈도우 레벨에서 합치기
# ---------------------------------------------------------------
X_windows = []
for (Xf, idxf) in X_list:
    Xf_scaled = scaler_mm.transform(Xf)                 # 공통 스케일 적용
    Xw, y, Xw_idx, y_idx = rolling_window_sequences(
        Xf_scaled, idxf, window_size=win_size,
        target_size=1, step_size=1, target_column=0)
    X_windows.append(Xw)

X_all = np.concatenate(X_windows, axis=0)               # 윈도우 합치기
print("최종 학습 윈도우:", X_all.shape)   # (윈도우 수, win_size, features_dim) — 파일 수만큼 커짐

# ---------------------------------------------------------------
# 6. 학습 (기존 루프 그대로 — 셔플이 51개 파일을 골고루 섞어줌)
# ---------------------------------------------------------------
X_ = np.copy(X_all)
for epoch in range(1, epochs+1):
    np.random.shuffle(X_)      # 매 epoch 셔플 → 매 배치에 51개 파일 골고루
    # ... 기존 critic/generator 학습 루프 그대로 ...

# ---------------------------------------------------------------
# 7. PCA / scaler 저장 (테스트가 같은 좌표계 쓰도록 — consistent 모드)
# ---------------------------------------------------------------
joblib.dump(scaler_std, 'checkpoints/scaler_std.joblib')
joblib.dump(pca, 'checkpoints/pca.joblib')
joblib.dump(scaler_mm, 'checkpoints/scaler_mm.joblib')
# 테스트(03_test)에서 이 셋을 로드해 transform만 하면 좌표계 일치